## Script for calculating OFH participants' PRS for large PRS files (100,000+ variants)
*Callum Webb*

Unlike GRS_script.ipynb, this script is built to deal with PRS of larger SNP quantities (tens of thousands of SNPs) rather than hundreds.

Running this script requires the following programs:

- plink2
- bgenix

You can set filepaths in the preferences chunk below, then run the rest of the script.

This script does **NOT** require a spark cluster. But you definitely want an instance with 32GB RAM or higher.

In [ ]:
# User preferences

## PRS scoring file - set filepath and column names

PRS_scoring_file <- "project-J8PJg1V25y16J0Y3F05pPJBj:/users/callum/GRS/David/PGS005395_hmPOS_GRCh38.txt"
    chr_column <- "hm_chr"
    pos_column <- "hm_pos"
    ref_column <- "other_allele"
    alt_column <- "effect_allele"
    score_column <- "effect_weight"

## Directories (if in doubt, leave unchanged)

programs_directory <- "project-J8PJg1V25y16J0Y3F05pPJBj:/programs/"
bgen_directory <- "project-J8PJg1V25y16J0Y3F05pPJBj:/imputed_bgen/"

In [ ]:
#####################################################
#==== Load packages and set up helper functions ====#
#####################################################

required_packages <- c("data.table")

for (pkg in required_packages) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg)
  }#
}

run_command <- function(command, args) {
  shell_command <- paste(
    c(shQuote(command, type = "sh"), vapply(args, shQuote, character(1), type = "sh")),
    collapse = " "
  )
  bash_command <- paste(
    "unset HTTP_PROXY HTTPS_PROXY http_proxy https_proxy;",
    shell_command
  )

  result <- system2(
    "bash",
    args = c("-c", shQuote(bash_command, type = "sh")),
    stdout = TRUE,
    stderr = TRUE
  )

  status <- attr(result, "status")
  if (is.null(status)) status <- 0L

  if (status != 0L) {
    stop(
      command, " command failed:\n",
      paste(result, collapse = "\n"),
      call. = FALSE
    )
  }

  result
}

run_dx <- function(args) run_command("dx", args)

## Part 1: Calculate PRS for OFH participants

Runtime: 7-8 hours on a 32GB instance

In [ ]:
#######################################
#== Step 1 - Fetch PRS scoring file ==#
#######################################

# Download PLINK2 if necessary
if (!file.exists("plink2")) {
  run_dx(c(
    "download",
    paste0(programs_directory, "plink2")
  ))
}

if (!file.exists("plink2")) {
  stop("plink2 was not downloaded from the programs directory.")
}

# Make the downloaded PLINK2 binary executable
plink2_bin <- normalizePath(
  "plink2",
  mustWork = TRUE
)

Sys.chmod(
  plink2_bin,
  mode = "0755"
)

# Confirm executable permissions
plink2_info <- file.info(plink2_bin)

message("Using PLINK2 at: ", plink2_bin)
message(
  "PLINK2 permissions: ",
  as.character(plink2_info$mode)
)

# Test the executable before processing any BGEN shards
plink2_test <- system2(
  plink2_bin,
  "--version",
  stdout = TRUE,
  stderr = TRUE
)

plink2_status <- attr(plink2_test, "status")

if (!is.null(plink2_status) && plink2_status != 0L) {
  cat(
    "---- PLINK2 executable test ----\n",
    paste(plink2_test, collapse = "\n"),
    "\n",
    sep = ""
  )

  stop(
    sprintf(
      "PLINK2 exists but could not be executed; exit status %d.",
      plink2_status
    )
  )
}

message(
  "PLINK2 executable test passed: ",
  paste(plink2_test, collapse = " ")
)

# Fetch PRS scoring file into the current JupyterLab session

if (!file.exists(basename(PRS_scoring_file))) {
  run_dx(c("download", paste0(PRS_scoring_file)))
}

scoring_file   <- basename(PRS_scoring_file)

stopifnot(
  file.exists(scoring_file)
)

# Make a one-column PRS variant-ID list

PRS_loadings <- data.table::fread(
  scoring_file,
  data.table = TRUE
)

required_columns <- c(chr_column, pos_column, ref_column, alt_column)

missing_columns <- setdiff(
  required_columns,
  names(PRS_loadings)
)

if (length(missing_columns)) {
  stop(
    sprintf(
      "PRS loadings file is missing required column(s): %s",
      paste(missing_columns, collapse = ", ")
    )
  )
}

# Create ID column in format chrX:position:REF:ALT
PRS_loadings[, ID := paste(
  PRS_loadings[[chr_column]],
  PRS_loadings[[pos_column]],
  PRS_loadings[[ref_column]],
  PRS_loadings[[alt_column]],
  sep = ":"
)]

id_parts <- data.table::tstrsplit(
  PRS_loadings$ID,
  ":",
  fixed = TRUE
)

if (length(id_parts) != 4L) {
  stop(
    paste0(
      "The PRS loading variant IDs do not consistently have four ",
      "colon-separated fields: chr, position, REF, ALT."
    )
  )
}

PRS_loadings[, CHROM := sub(
  "^chr",
  "",
  id_parts[[1]],
  ignore.case = TRUE
)]

PRS_loadings[, POS := id_parts[[2]]]
PRS_loadings[, REF_FROM_ID := toupper(id_parts[[3]])]
PRS_loadings[, ALT_FROM_ID := toupper(id_parts[[4]])]
PRS_loadings[, ALT := toupper(get(alt_column))]

## validate the ID against the effect_allele column

alt_mismatch <- PRS_loadings[
  !is.na(get(alt_column)) &
  !is.na(ALT_FROM_ID) &
  toupper(get(alt_column)) != ALT_FROM_ID
]

if (nrow(alt_mismatch)) {
  cat(
    "Examples of ID/effect_allele mismatches:\n"
  )

  print(
    head(
      alt_mismatch[, .(
        ID,
        ALT,
        REF_FROM_ID,
        ALT_FROM_ID
      )],
      10L
    )
  )

  stop(sprintf(
    paste0(
      "%d loading rows have an effect_allele column that does not match ",
      "the fourth component of ID."
    ),
    nrow(alt_mismatch)
  ))
}

message(
  "PRS loading IDs are consistent with chr:position:REF:ALT."
)

# Construct both possible orientations of the PRS loadings for PLINK2 scoring

# OFH-style CHROM-POS-REF-ALT
PRS_loadings[, OFH_ID_REF_ALT := paste(
  CHROM,
  POS,
  REF_FROM_ID,
  ALT_FROM_ID,
  sep = "-"
)]

# Also include CHROM-POS-ALT-REF because some OFH entries may use
# the opposite allele order
PRS_loadings[, OFH_ID_ALT_REF := paste(
  CHROM,
  POS,
  ALT_FROM_ID,
  REF_FROM_ID,
  sep = "-"
)]

ofh_extraction_ids <- unique(
  c(
    PRS_loadings$OFH_ID_REF_ALT,
    PRS_loadings$OFH_ID_ALT_REF
  )
)

ofh_extraction_ids <- ofh_extraction_ids[
  !is.na(ofh_extraction_ids) &
  nzchar(ofh_extraction_ids)
]

if (!length(ofh_extraction_ids)) {
  stop(
    "No OFH-formatted positional identifiers were generated."
  )
}

writeLines(
  ofh_extraction_ids,
  "variants.ofh_id"
)

message(sprintf(
  paste0(
    "Wrote %d unique OFH-formatted identifiers ",
    "from %d PRS scoring variants."
  ),
  length(ofh_extraction_ids),
  data.table::uniqueN(PRS_loadings$ID)
))

message("First OFH extraction identifiers:")

print(
  head(
    ofh_extraction_ids,
    10L
  )
)

if (!score_column %in% names(PRS_loadings)) {
  stop(sprintf(
    "PRS scoring file is missing the score column: %s",
    score_column
  ))
}

PRS_loadings[, EFFECT_ALLELE := toupper(get(alt_column))]
PRS_loadings[, EFFECT_WEIGHT := suppressWarnings(as.numeric(get(score_column)))]

if (anyNA(PRS_loadings$EFFECT_WEIGHT)) {
  stop("PRS scoring weights contain missing or non-numeric values.")
}

if (anyDuplicated(PRS_loadings$ID)) {
  stop("PRS scoring file contains duplicate variant IDs.")
}

In [ ]:
#######################################################################
# Step 2A — Subset OFH BGENs to variants in the PRS scoring file        #
# (bgenix subset → index → pgen)                                      #
#######################################################################

stopifnot(
  file.exists(scoring_file),
  file.exists("variants.ofh_id"),
  file.size("variants.ofh_id") > 0
)

message("First PRS variant IDs in extraction file:")

print(
  head(
    readLines("variants.ofh_id"),
    10
  )
)

### Load in bgenix 

if (!file.exists("bgenix")) {
  run_dx(c("download", paste0(programs_directory, "bgenix")))
}

if (!file.exists("bgenix")) stop("bgenix was not downloaded from the programs directory.")

# The downloaded file is the executable itself, not a source archive.
bgenix_bin <- normalizePath("bgenix", mustWork = TRUE)
Sys.chmod(bgenix_bin, "0755")
Sys.setenv(PATH = paste(dirname(bgenix_bin), Sys.getenv("PATH"), sep = ":"))

bgenix_on_path <- Sys.which("bgenix")
if (!nzchar(bgenix_on_path)) stop("bgenix is not executable or is not available on PATH.")
message("Using bgenix at ", bgenix_on_path)
system2(bgenix_on_path, "-help")


# Helper: choose a writable work directory with an actual write test
choose_writable_dir <- function(candidates) {
  # return first path for which we can create+delete a file
  for (d in candidates) {
    if (!nzchar(d)) next
    ok <- try(dir.create(d, recursive = TRUE, showWarnings = FALSE), silent = TRUE)
    if (!dir.exists(d)) next
    tf <- file.path(d, sprintf(".writetest_%s", as.integer(runif(1, 1e6, 9e6))))
    can <- tryCatch({ file.create(tf) }, warning = function(e) FALSE, error = function(e) FALSE)
    if (isTRUE(can)) { file.remove(tf); return(normalizePath(d, mustWork = TRUE)) }
  }
  return(NA_character_)
}

# Prefer project subfolders if writable; otherwise fall back to CWD/home/tempdir.
job_tag <- Sys.getenv("DX_JOB_ID", unset = format(Sys.time(), "%Y%m%d_%H%M%S"))
work_dir <- choose_writable_dir(c(
  file.path("/mnt/project", "work", paste0("pca_subset_", job_tag)),
  file.path("/mnt/project", paste0("pca_subset_", job_tag)),
  file.path(getwd(),       paste0("pca_subset_", job_tag)),
  file.path("~",           paste0("pca_subset_", job_tag)),
  tempdir()  # last resort
))
if (is.na(work_dir)) {
  stop("Could not find a writable work directory. Check project permissions.")
}
message("Using work_dir: ", work_dir)

# For downstream parts, reuse 'work_dir' consistently instead of Sys.getenv("TMPDIR")

tmp_dir <- work_dir

message("Using tmp_dir: ", tmp_dir)

# Ensure tmp_dir exists and is writable; if not, fall back to a fresh temp path

dir.create(tmp_dir, recursive = TRUE, showWarnings = FALSE)

.testfile <- file.path(tmp_dir, sprintf(".writetest_%s", as.integer(Sys.time())))
.canwrite <- tryCatch(isTRUE(file.create(.testfile)),
                      warning = function(e) FALSE, error = function(e) FALSE)
if (.canwrite) unlink(.testfile, force = TRUE) else {
  job_tag <- Sys.getenv("DX_JOB_ID", unset = format(Sys.time(), "%Y%m%d_%H%M%S"))
  tmp_dir <- file.path(tempdir(), paste0("pca_subset_", job_tag))
  dir.create(tmp_dir, recursive = TRUE, showWarnings = FALSE)
  
  .testfile <- file.path(tmp_dir, sprintf(".writetest_%s", as.integer(Sys.time())))
  .canwrite <- tryCatch(isTRUE(file.create(.testfile)),
                        warning = function(e) FALSE, error = function(e) FALSE)
  if (.canwrite) unlink(.testfile, force = TRUE) else {
    stop("No writable tmp_dir available. Check filesystem permissions.")
  }
}
message("Final tmp_dir (writable): ", tmp_dir)

# Define per‑loop helpers here so they’re in scope

extract_file <- normalizePath(
  "variants.ofh_id"
)

subset_prefixes <- character()

threads <- "16"
mem_gb <- "64"

all_files <- run_dx(c("ls", bgen_directory))

read_bgen_variant_count <- function(bgen_file) {
  con <- file(
    bgen_file,
    open = "rb"
  )

  on.exit(close(con))

  header_values <- readBin(
    con,
    what = integer(),
    n = 4L,
    size = 4L,
    endian = "little"
  )

  if (length(header_values) != 4L) {
    stop(
      "Could not read the BGEN header from: ",
      bgen_file
    )
  }

  # BGEN header values:
  # 1. offset
  # 2. header-block length
  # 3. number of variants
  # 4. number of samples
  as.numeric(header_values[3L])
}

for (chr in 1:22) {

  chr_bgens <- sort(grep(
    paste0(
      "^ofh_imputed\\.v6\\.chr",
      chr,
      "-b[0-9]+\\.bgen$"
    ),
    all_files,
    value = TRUE
  ))

  message(sprintf(
    "Chromosome %d: %d shards found",
    chr,
    length(chr_bgens)
  ))

  if (!length(chr_bgens)) {
    warning(sprintf(
      "No BGEN shards found for chromosome %d",
      chr
    ))
    next
  }

  message(sprintf(
    "Full run: processing all %d chromosome %d shards",
    length(chr_bgens),
    chr
  ))

for (src_bgen_name in chr_bgens) {

  src_sample_name <- sub(
    "\\.bgen$",
    ".sample",
    src_bgen_name
  )

  src_bgi_name <- paste0(
    src_bgen_name,
    ".bgi"
  )

  shard_name <- sub(
    "\\.bgen$",
    "",
    basename(src_bgen_name)
  )

  # Put downloaded source files in tmp_dir
  local_bgen <- file.path(
    tmp_dir,
    basename(src_bgen_name)
  )

  local_bgi <- paste0(
    local_bgen,
    ".bgi"
  )

  local_sample <- file.path(
    tmp_dir,
    basename(src_sample_name)
  )

  message(
    sprintf(
      "[chr %d] Downloading source files for %s",
      chr,
      shard_name
    )
  )

  # Remove files left by an earlier interrupted attempt
  unlink(
    c(
      local_bgen,
      local_bgi,
      local_sample
    ),
    force = TRUE
  )

  run_dx(c(
    "download",
    paste0(bgen_directory, src_bgen_name),
    "-o",
    local_bgen
  ))

  run_dx(c(
    "download",
    paste0(bgen_directory, src_bgi_name),
    "-o",
    local_bgi
  ))

  run_dx(c(
    "download",
    paste0(bgen_directory, src_sample_name),
    "-o",
    local_sample
  ))

  required_source_files <- c(
    local_bgen,
    local_bgi,
    local_sample
  )

  missing_source_files <- required_source_files[
    !file.exists(required_source_files)
  ]

  if (length(missing_source_files)) {
    stop(
      sprintf(
        "Failed to download required source file(s) for %s: %s",
        shard_name,
        paste(missing_source_files, collapse = ", ")
      )
    )
  }

## check first few records from the downloaded chromosome shard

if (chr == 1L && identical(src_bgen_name, chr_bgens[1L])) {

  preview_cmd <- sprintf(
    "%s -g %s -list 2>&1 | head -n 20",
    shQuote(bgenix_on_path),
    shQuote(local_bgen)
  )

  raw_index_preview <- system(
    preview_cmd,
    intern = TRUE
  )

  cat(
    "---- First unfiltered OFH BGEN index lines ----\n",
    paste(raw_index_preview, collapse = "\n"),
    "\n",
    sep = ""
  )

  cat(
    "---- First requested PRS positional IDs ----\n",
    paste(
      head(readLines(extract_file), 10L),
      collapse = "\n"
    ),
    "\n",
    sep = ""
  )
}

## Precheck: determine exactly how many requested variants are in this shard

match_out <- suppressWarnings(system2(
  bgenix_on_path,
  c(
    "-g", local_bgen,
    "-list",
    "-incl-rsids", extract_file
  ),
  stdout = TRUE,
  stderr = TRUE
))

match_status <- attr(match_out, "status")

if (!is.null(match_status) && match_status != 0L) {
  cat(
    "---- bgenix precheck output ----\n",
    paste(match_out, collapse = "\n"),
    "\n",
    sep = ""
  )

  stop(sprintf(
    "bgenix precheck failed for chr %d, shard %s.",
    chr,
    shard_name
  ))
}

# bgenix reports a line resembling:
# "# bgenix: success, total 0 variants."

summary_line <- grep(
  "total[[:space:]]+[0-9]+[[:space:]]+variants",
  match_out,
  value = TRUE
)

if (!length(summary_line)) {
  cat(
    "---- Unexpected bgenix precheck output ----\n",
    paste(match_out, collapse = "\n"),
    "\n",
    sep = ""
  )

  stop(sprintf(
    paste0(
      "Could not determine the number of matching variants ",
      "for chr %d, shard %s."
    ),
    chr,
    shard_name
  ))
}

# Use the final matching summary line in case bgenix emitted more than one
summary_line <- tail(summary_line, 1L)

n_matching_variants <- as.integer(sub(
  ".*total[[:space:]]+([0-9]+)[[:space:]]+variants.*",
  "\\1",
  summary_line
))

if (is.na(n_matching_variants)) {
  cat(
    "---- Unparseable bgenix summary ----\n",
    summary_line,
    "\n",
    sep = ""
  )

  stop(sprintf(
    "Could not parse variant count for chr %d, shard %s.",
    chr,
    shard_name
  ))
}

message(sprintf(
  "[chr %d] %s: %d requested variants found",
  chr,
  shard_name,
  n_matching_variants
))

if (n_matching_variants == 0L) {
  message(sprintf(
    "[chr %d] No requested variants found in %s; skipping.",
    chr,
    shard_name
  ))

  unlink(
    c(
      local_bgen,
      local_bgi,
      local_sample
    ),
    force = TRUE
  )

  next
}

    subset_bgen <- file.path(
        tmp_dir,
        paste0(shard_name, "_subset.bgen")
    )

    unlink(
      c(
        subset_bgen,
        paste0(subset_bgen, ".bgi")
      ),
      force = TRUE
    )
    
    subset_err <- paste0(
      subset_bgen,
      ".stderr"
    )

    unlink(
      subset_err,
      force = TRUE
    )

    cmd_subset <- sprintf(
      "%s -g %s -incl-rsids %s > %s 2> %s",
      shQuote(bgenix_on_path),
      shQuote(local_bgen),
      shQuote(extract_file),
      shQuote(subset_bgen),
      shQuote(subset_err)
    )

    message(sprintf(
      "[chr %d] Extracting %d variants from %s",
      chr,
      n_matching_variants,
      shard_name
    ))

    st1 <- system(cmd_subset)

    if (
      st1 != 0L ||
      !file.exists(subset_bgen) ||
      file.size(subset_bgen) == 0
    ) {
      if (file.exists(subset_err)) {
        cat(
          "---- bgenix stderr ----\n",
          paste(
            readLines(subset_err, warn = FALSE),
            collapse = "\n"
          ),
          "\n",
          sep = ""
        )
      }

      stop(sprintf(
        "bgenix extraction failed for chr %d, shard %s.",
        chr,
        shard_name
      ))
    }

    # A BGEN containing zero variants still has a nonzero-sized header,
    # so inspect the variant count explicitly.
    subset_variant_count <- read_bgen_variant_count(
      subset_bgen
    )

    message(sprintf(
      "[chr %d] %s: extracted BGEN header reports %d variants",
      chr,
      shard_name,
      subset_variant_count
    ))

    if (subset_variant_count == 0L) {
      message(sprintf(
        "[chr %d] Extracted BGEN for %s is empty; skipping.",
        chr,
        shard_name
      ))

      unlink(
        c(
          local_bgen,
          local_bgi,
          local_sample,
          subset_bgen,
          subset_err
        ),
        force = TRUE
      )

      next
    }

    if (subset_variant_count != n_matching_variants) {
      warning(sprintf(
        paste0(
          "Variant-count mismatch for chr %d, shard %s: ",
          "precheck found %d but extracted BGEN reports %d."
        ),
        chr,
        shard_name,
        n_matching_variants,
        subset_variant_count
      ))
    }
    
    # Convert the small BGEN to PGEN (use the staged .sample)
    pfx <- file.path(
        tmp_dir,
        paste0(shard_name, "_subset")
    )

    unlink(
      paste0(
        pfx,
        c(
          ".pgen",
          ".pvar",
          ".psam",
          ".log"
        )
      ),
      force = TRUE
    )

    args_make_pgen <- c(
      "--threads", threads,
      "--memory",  as.character(as.integer(mem_gb) * 1024),
      "--bgen",    subset_bgen, "ref-first",
      "--sample",  local_sample,
      "--make-pgen",
      "--out",     pfx
    )
    
    message(
      sprintf(
        "[chr %d] %s -> PGEN",
        chr,
        basename(src_bgen_name)
      )
    )

out_txt <- system2(
  plink2_bin,
  args_make_pgen,
  stdout = TRUE,
  stderr = TRUE
)

required_pgen_files <- paste0(
  pfx,
  c(".pgen", ".pvar", ".psam")
)

if (!all(file.exists(required_pgen_files))) {
  logf <- paste0(pfx, ".log")

  if (file.exists(logf)) {
    cat(
      "---- PLINK log tail ----\n",
      paste(tail(readLines(logf), 50), collapse = "\n"),
      "\n",
      sep = ""
    )
  }

  cat(
    "---- Captured PLINK output ----\n",
    paste(tail(out_txt, 50), collapse = "\n"),
    "\n",
    sep = ""
  )

  stop(
    sprintf(
      "PLINK --make-pgen failed for chr %d, shard %s.",
      chr,
      shard_name
    )
  )
}

    # Only record a successfully completed PGEN fileset
    subset_prefixes <- c(
      subset_prefixes,
      pfx
    )

    # Reclaim disk space before downloading the next source shard
    unlink(
      c(
        local_bgen,
        local_bgi,
        local_sample,
        subset_bgen,
        paste0(subset_bgen, ".bgi"),
        subset_err
      ),
      force = TRUE
    )

  } # end shard loop
} # end chromosome loop

# Optional: you can delete subset_bgen + .bgi now to reclaim space
# unlink(c(subset_bgen, paste0(subset_bgen, ".bgi")), force = TRUE)

system("df -h")
system(sprintf("df -h %s", shQuote(work_dir)))

## count total pgens produced

message(sprintf(
  "Total shard-level PGENs produced: %d",
  length(subset_prefixes)
))

if (anyDuplicated(subset_prefixes)) {
  duplicated_prefixes <- unique(
    subset_prefixes[duplicated(subset_prefixes)]
  )

  stop(sprintf(
    "Duplicate shard-level PGEN prefixes detected: %s",
    paste(duplicated_prefixes, collapse = ", ")
  ))
}




In [ ]:
# ===================================== #
# Step 2B — Merge to a single `.pfile`  #
# ===================================== #

merge_list <- file.path(tmp_dir, "pmerge_list.txt")

# Which per-chr pfiles exist?

complete_fileset <- vapply(
  subset_prefixes,
  function(pfx) {
    all(file.exists(paste0(
      pfx,
      c(".pgen", ".pvar", ".psam")
    )))
  },
  logical(1)
)

existing <- which(complete_fileset)

if (!length(existing)) {
  stop(
    "No complete shard-level PGEN filesets were produced. Check Step 2A logs."
  )
}

if (any(!complete_fileset)) {
  warning(sprintf(
    "%d shard-level PGEN prefixes are incomplete and will be excluded.",
    sum(!complete_fileset)
  ))
}

if (length(existing) == 1L) {
  stop(
    paste0(
      "Only one shard-level PGEN was produced. ",
      "A merge is unnecessary; inspect the shard-processing results."
    )
  )
}

# Choose the first existing prefix as the merge base

base_idx    <- existing[1]
base_prefix <- subset_prefixes[base_idx]

# Everything else (excluding the base) goes into --pmerge-list

merge_targets <- subset_prefixes[setdiff(existing, base_idx)]
writeLines(merge_targets, con = merge_list)

merged_pfx <- file.path(tmp_dir, "ofh_subset_merged")

args_merge <- c(
  "--threads", threads,
  "--memory",  as.character(as.integer(mem_gb) * 1024),
  "--pfile",   base_prefix,        # <- now guaranteed to exist
  "--pmerge-list", merge_list,
  "--out",     merged_pfx
)

## remove any stale outputs

unlink(
  paste0(
    merged_pfx,
    c(".pgen", ".pvar", ".psam", ".log")
  ),
  force = TRUE
)

## run merge

message(sprintf(
  "Merging %d shard-level PGEN filesets into one genome-wide PGEN",
  length(existing)
))

merge_out <- system2(
  plink2_bin,
  args_merge,
  stdout = TRUE,
  stderr = TRUE
)


## check PLINK exit status and that the merged files exist

merge_status <- attr(
  merge_out,
  "status"
)

required_merged_files <- paste0(
  merged_pfx,
  c(".pgen", ".pvar", ".psam")
)

if (
  (!is.null(merge_status) && merge_status != 0L) ||
  !all(file.exists(required_merged_files))
) {
  cat(
    "---- PLINK merge output ----\n",
    paste(tail(merge_out, 100), collapse = "\n"),
    "\n",
    sep = ""
  )

  merge_log <- paste0(
    merged_pfx,
    ".log"
  )

  if (file.exists(merge_log)) {
    cat(
      "---- PLINK merge log tail ----\n",
      paste(tail(readLines(merge_log), 100), collapse = "\n"),
      "\n",
      sep = ""
    )
  }

  stop("PLINK shard merge failed.")
}



# This `--pmerge-list` route is the documented way to merge PLINK2 pfiles. [2](https://zenodo.org/records/15420125)


In [ ]:
# ========================================================== #
# Step 2C - Calculate the PRS for OFH participants           #
# ========================================================== #

library(data.table)

# Match the canonical PRS variants to the IDs retained in the merged PVAR.
ofh_pvar <- fread(paste0(merged_pfx, ".pvar"), data.table = TRUE)
required_pvar_columns <- c("#CHROM", "POS", "ID", "REF", "ALT")
missing_pvar_columns <- setdiff(required_pvar_columns, names(ofh_pvar))
if (length(missing_pvar_columns)) {
  stop(sprintf(
    "Merged PVAR is missing required column(s): %s",
    paste(missing_pvar_columns, collapse = ", ")
  ))
}

ofh_pvar[, CHROM_KEY := sub("^chr", "", as.character(get("#CHROM")), ignore.case = TRUE)]
ofh_pvar[, POS_KEY := as.character(POS)]
ofh_pvar[, REF_KEY := toupper(as.character(REF))]
ofh_pvar[, ALT_KEY := toupper(as.character(ALT))]
ofh_pvar[, VARIANT_KEY := paste(CHROM_KEY, POS_KEY,
  pmin(REF_KEY, ALT_KEY), pmax(REF_KEY, ALT_KEY), sep = ":"
)]

prs_variants <- PRS_loadings[
  ,
  .(ID, CHROM, POS, REF = REF_FROM_ID, ALT = ALT_FROM_ID,
    EFFECT_ALLELE, EFFECT_WEIGHT)
]
prs_variants[, VARIANT_KEY := paste(CHROM, POS,
  pmin(REF, ALT), pmax(REF, ALT), sep = ":"
)]

if (anyDuplicated(ofh_pvar$VARIANT_KEY)) {
  stop("Merged PVAR contains duplicate chromosome-position-allele keys.")
}
if (anyDuplicated(prs_variants$VARIANT_KEY)) {
  stop("PRS scoring file contains duplicate chromosome-position-allele keys.")
}

score_variants <- merge(
  prs_variants,
  ofh_pvar[, .(VARIANT_KEY, SCORE_ID = as.character(ID),
               OFH_REF = REF_KEY, OFH_ALT = ALT_KEY)],
  by = "VARIANT_KEY", all = FALSE
)

if (!nrow(score_variants)) {
  stop("No PRS variants could be matched to the merged OFH PGEN.")
}

if (any(!score_variants$EFFECT_ALLELE %in%
        c(score_variants$OFH_REF, score_variants$OFH_ALT))) {
  stop("Some PRS effect alleles are incompatible with the matched OFH variants.")
}

prs_score_file <- file.path(tmp_dir, "OFH_PRS_scoring.tsv")
fwrite(
  score_variants[, .(ID = SCORE_ID, ALLELE = EFFECT_ALLELE,
                     EFFECT_WEIGHT)],
  prs_score_file, sep = "\t", quote = FALSE
)

score_out <- file.path(tmp_dir, "OFH_PRS")
unlink(paste0(score_out, c(".sscore", ".sscore.vars", ".log")), force = TRUE)

args_score <- c(
  "--threads", threads,
  "--memory", as.character(as.integer(mem_gb) * 1024),
  "--pfile", merged_pfx,
  "--score", prs_score_file, "1", "2", "3",
  "header-read", "cols=+scoresums", "list-variants",
  "--out", score_out
)

message(sprintf(
  "Calculating PRS using %d variants retained in the merged OFH PGEN",
  nrow(score_variants)
))
score_log <- system2(plink2_bin, args_score, stdout = TRUE, stderr = TRUE)
score_status <- attr(score_log, "status")
required_score_files <- paste0(score_out, c(".sscore", ".sscore.vars"))
if ((!is.null(score_status) && score_status != 0L) ||
    !all(file.exists(required_score_files))) {
  cat("---- PLINK PRS output ----\n", paste(tail(score_log, 100), collapse = "\n"), "\n", sep = "")
  stop("PLINK PRS scoring failed.")
}

ss <- fread(paste0(score_out, ".sscore"), data.table = TRUE)

# Use non-missing allele-normalised PRS from the scoring file
prs_sum_column <- intersect(
  c("EFFECT_WEIGHT_AVG", "EFFECT_WEIGHT_SUM"),
  names(ss)
)[1]
if (is.na(prs_sum_column)) {
  stop(
    "PLINK score output does not contain EFFECT_WEIGHT_SUM or EFFECT_WEIGHT_AVG."
  )
}

prs_output <- ss[, .(IID, PRS = get(prs_sum_column))]
prs_output_file <- file.path(tmp_dir, "OFH_PRS.csv")
fwrite(prs_output, prs_output_file)

prs_audit <- data.table(
  scoring_file_variants = nrow(PRS_loadings),
  variants_matched_to_OFH = nrow(score_variants),
  retention_rate = nrow(score_variants) / nrow(PRS_loadings)
)
prs_audit_file <- file.path(tmp_dir, "OFH_PRS_audit.tsv")
fwrite(prs_audit, prs_audit_file, sep = "\t", quote = FALSE)

files_to_upload <- c(
  paste0(merged_pfx, c(".pgen", ".pvar", ".psam", ".log")),
  prs_score_file,
  paste0(score_out, c(".sscore", ".sscore.vars", ".log")),
  prs_output_file,
  prs_audit_file
)

for (file_to_upload in unique(files_to_upload)) {
  if (!file.exists(file_to_upload)) {
    warning("Upload skipped because file is missing: ", file_to_upload)
    next
  }
  run_dx(c("upload", file_to_upload))
}
